In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('../../01-intro/1.9 Bike Sharing/data/day.csv')
df['dteday'] = pd.to_datetime(df['dteday'])
df = df.sort_values('dteday').reset_index(drop=True)

assert df['dteday'].is_monotonic_increasing
assert df.shape[0] == 731

print(df.head(1))
print(df.tail(1))

   instant     dteday  season  yr  mnth  holiday  weekday  workingday  \
0        1 2011-01-01       1   0     1        0        6           0   

   weathersit      temp     atemp       hum  windspeed  casual  registered  \
0           2  0.344167  0.363625  0.805833   0.160446     331         654   

   cnt  
0  985  
     instant     dteday  season  yr  mnth  holiday  weekday  workingday  \
730      731 2012-12-31       1   1    12        0        1           1   

     weathersit      temp     atemp     hum  windspeed  casual  registered  \
730           2  0.215833  0.223487  0.5775   0.154846     439        2290   

      cnt  
730  2729  


In [2]:
n = df.shape[0]
n_train = int(n * 0.7)
n_val = int(n * 0.15)
n_test = int(n * 0.15)

df_train = df.iloc[:n_train]
df_val = df.iloc[n_train:n_train+n_val]
df_test = df.iloc[n_train+n_val:]

print(df_train.shape, df_val.shape, df_test.shape)


(511, 16) (109, 16) (111, 16)


In [3]:
assert df_train['dteday'].max() < df_val['dteday'].min()
assert df_val['dteday'].max() < df_test['dteday'].min()

print(df_train['dteday'].max())
print(df_val['dteday'].min())
print(df_val['dteday'].max())
print(df_test['dteday'].min())

2012-05-25 00:00:00
2012-05-26 00:00:00
2012-09-11 00:00:00
2012-09-12 00:00:00


In [4]:
assert df_train.shape[0] + df_val.shape[0] + df_test.shape[0] == n
df2 = pd.concat([df_train, df_val, df_test])
assert df.equals(df2)

In [5]:
train_mean = df_train['cnt'].mean()
val_pred_mean = np.full(df_val.shape[0], train_mean)
print(train_mean, val_pred_mean[:5], len(val_pred_mean))

3794.211350293542 [3794.21135029 3794.21135029 3794.21135029 3794.21135029 3794.21135029] 109


In [6]:
y_val = df_val['cnt']
absolute_errors = np.abs(y_val - val_pred_mean)
mae_mean_baseline = absolute_errors.mean()
print(absolute_errors[:5])
print(mae_mean_baseline)

511    2741.78865
512    2796.78865
513    2248.78865
514    1948.78865
515    3060.78865
Name: cnt, dtype: float64
2950.8161726422377


In [7]:
season_means = df_train.groupby('season')['cnt'].mean()
print(season_means)

season
1    2651.411765
2    4698.196203
3    4464.361702
4    3664.460674
Name: cnt, dtype: float64


In [8]:
val_pred_season = df_val['season'].map(season_means)
print(val_pred_season.head())
print(len(val_pred_season))


511    4698.196203
512    4698.196203
513    4698.196203
514    4698.196203
515    4698.196203
Name: season, dtype: float64
109


In [9]:
assert val_pred_season.notna().all()
season_absolute_errors = np.abs(y_val - val_pred_season)
mae_season_baseline = season_absolute_errors.mean()
print(season_absolute_errors[:5])
print(mae_season_baseline)

511    1837.803797
512    1892.803797
513    1344.803797
514    1044.803797
515    2156.803797
dtype: float64
2235.467828136698


In [10]:
signed_errors = y_val - val_pred_season
mean_signed_error = signed_errors.mean()
print(mean_signed_error)

2224.888784014509


In [13]:
n_under = (signed_errors > 0).sum()
n_over = (signed_errors < 0).sum()
n_zero = (signed_errors == 0).sum()

print(n_under, n_over, n_zero)

assert n_under + n_over + n_zero == len(signed_errors)

107 2 0


## Day 9 — Chronological Validation Results
1. Dataset splitting: We partitioned the 731 observations so we can have a training, validation, and test dataset. We didn't shuffle them because we wanted them to be in chronological order so that our machine learning model doesn't accidentally get trained with information from days ahead of the days it is trying to predict. This would cause a future past leakage as the model would have access to future information which it could potentially exploit to predict the past.
2. Baseline comparison: The mae_mean_baseline was 2950.8161726422377 and the mae_season_baseline was 2235.467828136698. The seasonal baseline performed better as it had a lower Mean Absolute Error, which indicates that it's average error was less than the mae_mean_baseline error. 
3. Prediction bias: The 107 underpredictions and positive mean signed error tells us that the seasonal baseline most of the time underpredicts the rentals. This is likely because it is biased downward during the validation period.
4. Limitations: Our current validation results may not accurately reflect future real-world performance because the validation data set is made up of specific seasons which affects the mean baseline and seasonal baseline. Another reason is that our model on 2011-2012 historical data may not work well to patterns in 2026 or later because of distribution shift. This is where statisical patterns encountered in deployment may differ from those observed in training.